# SDSS Photometric Redshift Regression with KAN
---
This project attempts to replicate the dataset and results of D'Abrusco's paper using KANs as an alternative to MLPs.
It should be noted that while this code's dataset uses SDSS DR17 (2021) and the paper's dataset uses SDSS DR4 (2005), data handling (such as which inputs and what types of samples) remains identical.
#### D'Abrusco et al. (2007): https://arxiv.org/pdf/astro-ph/0703108


In [ ]:
!pip -q install pykan astroquery pandas numpy scikit-learn torch

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import os
from google.colab import drive
from astroquery.sdss import SDSS
from kan import KAN

# Set seed for fixed outputs
import random
random.seed(67)
np.random.seed(67)
torch.manual_seed(67)
torch.cuda.manual_seed_all(67)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation

Download and Process SDSS Data

Use paper's filters:
- z < 0.5
- dered_r < 21
- primary objects (mode=1)

Obtain 5 photometric inputs, plus objectID:
- dered_u
- dered_g
- dered_r
- dered_i
- dered_z

Target: spectroscopic redshift (z)

In [ ]:
# Query SDSS DR17 for required inputs
def run_sql(z_lo, z_hi, top_n):
    sql = f"""
    SELECT TOP {top_n}
        p.objID AS objid,
        s.z AS specz,
        (p.modelMag_u - p.extinction_u) AS dered_u,
        (p.modelMag_g - p.extinction_g) AS dered_g,
        (p.modelMag_r - p.extinction_r) AS dered_r,
        (p.modelMag_i - p.extinction_i) AS dered_i,
        (p.modelMag_z - p.extinction_z) AS dered_z,
        CASE
            WHEN s.z < 0.30 THEN 0
            WHEN s.z < 0.35 THEN 1
            WHEN s.z < 0.40 THEN 2
            WHEN s.z < 0.45 THEN 3
            ELSE 4
        END AS z_bin
    FROM SpecObj AS s
    JOIN PhotoObj AS p
        ON s.bestObjID = p.objID
    WHERE
        s.class = 'GALAXY'
        AND s.z IS NOT NULL
        AND s.z >= {z_lo} AND s.z < {z_hi}
        AND p.mode = 1
        AND s.zErr > 0 AND s.zErr < 0.015
        AND p.modelMag_u > -90 AND p.modelMag_g > -90
        AND p.modelMag_r > -90 AND p.modelMag_i > -90 AND p.modelMag_z > -90
        AND p.extinction_u > -90 AND p.extinction_g > -90
        AND p.extinction_r > -90 AND p.extinction_i > -90 AND p.extinction_z > -90
        AND (p.modelMag_r - p.extinction_r) BETWEEN 0 AND 21.0
        AND p.extinction_r < 0.30
        AND p.modelMagErr_u < 0.40
        AND p.modelMagErr_g < 0.20
        AND p.modelMagErr_r < 0.20
        AND p.modelMagErr_i < 0.20
        AND p.modelMagErr_z < 0.25
    ORDER BY z_bin ASC, p.objID ASC  -- Balances bins first, then sorts within
    """
    t = SDSS.query_sql(sql, data_release=17)
    return pd.DataFrame() if t is None else t.to_pandas()

In [ ]:
# Save/Load parquet files
drive.mount('drive')
# If you would like to use the parquet files saved, add the 2 parquet files from the repo to a new folder in your Google Drive titled "SDSS_DR17"
near_ds_file = 'drive/MyDrive/SDSS_DR17/near.parquet'
far_ds_file = 'drive/MyDrive/SDSS_DR17/far.parquet'

load_near = True
load_far = True

if load_near and os.path.exists(near_ds_file):
    df_near = pd.read_parquet(near_ds_file)
    print("Loaded pre-downloaded near dataset.")
else:
    print("Downloading near dataset...")
    df_near = run_sql(0.00, 0.27, 350000)
    print("New near dataset saved.")
    df_near.to_parquet(near_ds_file)


if load_far and os.path.exists(far_ds_file):
    df_far = pd.read_parquet(far_ds_file)
    print("Loaded pre-downloaded far dataset.")
else:
    print("Downloading far dataset...")
    df_far = run_sql(0.23, 0.50, 350000)
    print("New far dataset saved.")
    df_far.to_parquet(far_ds_file)

df = pd.concat([df_near, df_far], ignore_index=True)
df.columns = [c.lower() for c in df.columns]
df = df.drop_duplicates(subset=["objid"]).reset_index(drop=True)

print("total:", len(df))
print("near (<0.25):", (df["specz"] < 0.25).sum())
print("far (>=0.25):", (df["specz"] >= 0.25).sum())

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).
Loaded pre-downloaded near dataset.
Loaded pre-downloaded far dataset.
total: 463454
near (<0.25): 373174
far (>=0.25): 90280


In [ ]:
required = ['objid', 'specz', 'dered_u', 'dered_g', 'dered_r', 'dered_i', 'dered_z']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

# Remove invalids
df = df[required].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.drop_duplicates(subset=['objid']).reset_index(drop=True)

print('Rows after cleanup:', len(df))
print(df[['specz','dered_r']].describe())

Rows after cleanup: 463454
              specz        dered_r
count  4.634540e+05  463454.000000
mean   1.260512e-01      17.177790
std    1.116511e-01       1.207496
min    2.121375e-07       9.829470
25%    5.954332e-02      16.447593
50%    7.968576e-02      17.211695
75%    1.082102e-01      17.786487
max    4.999995e-01      20.999710


In [ ]:
# Split values obtained from paper
TARGET_TOTAL = 449_370
N_TRAIN = 269_622
N_TEST = 89_874

if len(df) < TARGET_TOTAL:
    raise RuntimeError(f'Need at least {TARGET_TOTAL} rows, got {len(df)}. Relax filters or rerun later.')

df_sample = df.sample(n=TARGET_TOTAL, random_state=int(random.random())).reset_index(drop=True)

train_df = df_sample.iloc[:N_TRAIN].reset_index(drop=True)
test_df  = df_sample.iloc[N_TRAIN : N_TRAIN + N_TEST].reset_index(drop=True)

print('Split sizes:')
print('train:', len(train_df))
print('test: ', len(test_df))

Split sizes:
train: 269622
test:  89874


## Split larger dataset into 2 subsets

Classifier gate:
- Near: z < 0.25
- Far: z >= 0.25

In [ ]:
# Build regressor subsets from EACH SPLIT SEPARATELY
feature_cols = ['dered_u', 'dered_g', 'dered_r', 'dered_i', 'dered_z']
target_col = 'specz'

near_train_df = train_df[(train_df[target_col] >= 0.00) & (train_df[target_col] <= 0.27)].copy()
near_test_df  = test_df[(test_df[target_col] >= 0.01) & (test_df[target_col] <= 0.25)].copy()

far_train_df  = train_df[(train_df[target_col] >= 0.23) & (train_df[target_col] <= 0.50)].copy()
far_test_df   = test_df[(test_df[target_col] >= 0.25) & (test_df[target_col] <= 0.48)].copy()

print('Regressor subset sizes:')
print('near_train:', len(near_train_df))
print('near_test :', len(near_test_df))
print('far_train :', len(far_train_df))
print('far_test  :', len(far_test_df))

Regressor subset sizes:
near_train: 227816
near_test : 71257
far_train : 65976
far_test  : 17212


In [ ]:
# Standardize features to allow efficient model training
def apply_reg_standardization(df_in, feature_cols, target_col, x_mu, x_sd, y_mu, y_sd):
    df_out = df_in.copy()
    df_out[feature_cols] = (df_out[feature_cols] - x_mu) / x_sd
    df_out[target_col] = (df_out[target_col] - y_mu) / y_sd
    return df_out

x_mu_n = near_train_df[feature_cols].mean()
x_sd_n = near_train_df[feature_cols].std(ddof=0).replace(0, 1.0)
y_mu_n = float(near_train_df[target_col].mean())
y_sd_n = float(near_train_df[target_col].std(ddof=0))

near_train_std = apply_reg_standardization(near_train_df, feature_cols, target_col, x_mu_n, x_sd_n, y_mu_n, y_sd_n)
near_test_std  = apply_reg_standardization(near_test_df,  feature_cols, target_col, x_mu_n, x_sd_n, y_mu_n, y_sd_n)

near_stats = {
    'feature_cols': feature_cols,
    'x_mean': x_mu_n.to_dict(),
    'x_std': x_sd_n.to_dict(),
    'y_mean': y_mu_n,
    'y_std': y_sd_n
}

x_mu_f = far_train_df[feature_cols].mean()
x_sd_f = far_train_df[feature_cols].std(ddof=0).replace(0, 1.0)
y_mu_f = float(far_train_df[target_col].mean())
y_sd_f = float(far_train_df[target_col].std(ddof=0))

far_train_std = apply_reg_standardization(far_train_df, feature_cols, target_col, x_mu_f, x_sd_f, y_mu_f, y_sd_f)
far_test_std  = apply_reg_standardization(far_test_df,  feature_cols, target_col, x_mu_f, x_sd_f, y_mu_f, y_sd_f)

far_stats = {
    'feature_cols': feature_cols,
    'x_mean': x_mu_f.to_dict(),
    'x_std': x_sd_f.to_dict(),
    'y_mean': y_mu_f,
    'y_std': y_sd_f
}

print('\nNear stats:')
print(json.dumps(near_stats, indent=2, default=float))
print('\nFar stats:')
print(json.dumps(far_stats, indent=2, default=float))

print(f"Near destandardization multiplier: {y_sd_n}")
print(f"Far destandardization multiplier: {y_sd_f}")


Near stats:
{
  "feature_cols": [
    "dered_u",
    "dered_g",
    "dered_r",
    "dered_i",
    "dered_z"
  ],
  "x_mean": {
    "dered_u": 19.19619838944587,
    "dered_g": 17.64003117204235,
    "dered_r": 16.890442001878707,
    "dered_i": 16.52116572334691,
    "dered_z": 16.273588757023212
  },
  "x_std": {
    "dered_u": 1.2046717549931583,
    "dered_g": 1.1105980898701604,
    "dered_r": 1.059103262520152,
    "dered_i": 1.0795574345474637,
    "dered_z": 1.1351473968362702
  },
  "y_mean": 0.08578506302870509,
  "y_std": 0.060349845087157156
}

Far stats:
{
  "feature_cols": [
    "dered_u",
    "dered_g",
    "dered_r",
    "dered_i",
    "dered_z"
  ],
  "x_mean": {
    "dered_u": 21.474308698769246,
    "dered_g": 19.917741408239362,
    "dered_r": 18.468111043409728,
    "dered_i": 17.91367537577301,
    "dered_z": 17.555960885124897
  },
  "x_std": {
    "dered_u": 0.8525250683038111,
    "dered_g": 0.7631690881654539,
    "dered_r": 0.7151070184077781,
    "dered_i": 

In [ ]:
# Create final dataset used by the KAN
near_dataset = {
    'train_input': torch.tensor(near_train_std[feature_cols].to_numpy(dtype=np.float32), device=device),
    'train_label': torch.tensor(near_train_std[[target_col]].to_numpy(dtype=np.float32), device=device),
    'test_input':  torch.tensor(near_test_std[feature_cols].to_numpy(dtype=np.float32), device=device),
    'test_label':  torch.tensor(near_test_std[[target_col]].to_numpy(dtype=np.float32), device=device),
}

far_dataset = {
    'train_input': torch.tensor(far_train_std[feature_cols].to_numpy(dtype=np.float32), device=device),
    'train_label': torch.tensor(far_train_std[[target_col]].to_numpy(dtype=np.float32), device=device),
    'test_input':  torch.tensor(far_test_std[feature_cols].to_numpy(dtype=np.float32), device=device),
    'test_label':  torch.tensor(far_test_std[[target_col]].to_numpy(dtype=np.float32), device=device),
}

print('Near:', near_dataset['train_input'].shape, near_dataset['test_input'].shape)
print('Far :', far_dataset['train_input'].shape, far_dataset['test_input'].shape)

Near: torch.Size([227816, 5]) torch.Size([71257, 5])
Far : torch.Size([65976, 5]) torch.Size([17212, 5])


# Creating and Training the KAN

In [ ]:
# Near Dataset
near_test_mae_list = []
near_test_rmse_list = []
sample = near_dataset["train_input"][:1024]

def train_mse():
    with torch.no_grad():
        predictions = model(near_dataset['train_input'])
        mse = torch.nn.functional.mse_loss(predictions, near_dataset['train_label'])
    return mse

def test_mse():
    with torch.no_grad():
        predictions = model(near_dataset['test_input'])
        mse = torch.nn.functional.mse_loss(predictions, near_dataset['test_label'])
    return mse

# Train 20 unique KANs
for datum in range(20):
    model = KAN(
        width=[5, 11, 5, 1],
        grid=5,
        k=4,
        seed=datum,
        device=device,
    )

    model.speed()
    model.fit(
        near_dataset,
        opt="Adam",
        update_grid=False,
        lr=5e-3,
        metrics=(train_mse, test_mse),
        loss_fn=torch.nn.MSELoss(),
        steps=1300
    )


    # Standardize back to values comparable to the paper by multiplying
    # MAE and RMSE by y_std, which was calculated earlier
    test_rmse = (test_mse() ** 0.5) * y_sd_n
    x = near_dataset["test_input"]
    y = near_dataset["test_label"].view(-1)
    pred = model(x).view(-1)
    test_err = pred - y
    test_mae = (torch.mean(torch.abs(test_err)).item()) * y_sd_n

    near_test_mae_list.append(test_mae)
    near_test_rmse_list.append(test_rmse)
    print(f"Appending following values: MAE={test_mae}, RMSE={test_rmse}")

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.86e-01 | test_loss: 2.78e-01 | reg: 0.00e+00 | : 100%|█| 1300/1300 [01:11<00:00, 18.


Appending following values: MAE=0.011764699567579438, RMSE=0.016790548339486122
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.86e-01 | test_loss: 2.79e-01 | reg: 0.00e+00 | : 100%|█| 1300/1300 [01:09<00:00, 18.


Appending following values: MAE=0.011809724857291148, RMSE=0.01681002415716648
checkpoint directory created: ./model
saving model version 0.0


...

Appending following values: MAE=0.011754290369520536, RMSE=0.016663677990436554
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.88e-01 | test_loss: 2.80e-01 | reg: 0.00e+00 | : 100%|█| 1300/1300 [01:11<00:00, 18.


Appending following values: MAE=0.011899878854233081, RMSE=0.01689853146672249
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.87e-01 | test_loss: 2.79e-01 | reg: 0.00e+00 | : 100%|█| 1300/1300 [01:11<00:00, 18.

Appending following values: MAE=0.011784311126220345, RMSE=0.01681649498641491


In [ ]:
# Far Dataset
far_test_mae_list = []
far_test_rmse_list = []

def train_mse():
    with torch.no_grad():
        predictions = model(far_dataset['train_input'])
        mse = torch.nn.functional.mse_loss(predictions, far_dataset['train_label'])
    return mse

def test_mse():
    with torch.no_grad():
        predictions = model(far_dataset['test_input'])
        mse = torch.nn.functional.mse_loss(predictions, far_dataset['test_label'])
    return mse

# Train 20 unique KANs
for datum in range(20):
    model = KAN(
        width=[5, 11, 5, 1],
        grid=5,
        k=4,
        seed=datum,
        device=device
    )

    model.speed()
    model.fit(
        far_dataset,
        opt="Adam",
        update_grid=False,
        lr=5e-3,
        metrics=(train_mse, test_mse),
        loss_fn=torch.nn.MSELoss(),
        steps=600,
    )


    # Standardize back to values comparable to the paper by multiplying
    # MAE and RMSE by y_std, which was calculated earlier
    test_rmse = (test_mse() ** 0.5) * y_sd_f
    x = far_dataset["test_input"]
    y = far_dataset["test_label"].view(-1)
    pred = model(x).view(-1)
    test_err = pred - y
    test_mae = (torch.mean(torch.abs(test_err)).item()) * y_sd_f

    far_test_mae_list.append(test_mae)
    far_test_rmse_list.append(test_rmse)
    print(f"Appending following values: MAE={test_mae}, RMSE={test_rmse}")

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.60e-01 | test_loss: 4.64e-01 | reg: 0.00e+00 | : 100%|█| 600/600 [00:13<00:00, 44.92


Appending following values: MAE=0.021624030445766624, RMSE=0.03146158158779144
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.57e-01 | test_loss: 4.62e-01 | reg: 0.00e+00 | : 100%|█| 600/600 [00:13<00:00, 44.87


Appending following values: MAE=0.021565837634815937, RMSE=0.03137390315532684
checkpoint directory created: ./model
saving model version 0.0


...

Appending following values: MAE=0.02160427566149121, RMSE=0.0314420722424984
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.57e-01 | test_loss: 4.62e-01 | reg: 0.00e+00 | : 100%|█| 600/600 [00:13<00:00, 44.79


Appending following values: MAE=0.02146959700017484, RMSE=0.031319908797740936
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.61e-01 | test_loss: 4.60e-01 | reg: 0.00e+00 | : 100%|█| 600/600 [00:13<00:00, 44.87


Appending following values: MAE=0.021537371012914617, RMSE=0.03123100847005844


## Evaluation Metrics

In [ ]:
# NEAR
# Convert RMSE tensors to CPU and then to Python numbers before calculating mean and std
near_test_rmse_list_cpu = [rmse.cpu().item() for rmse in near_test_rmse_list]
print(f"Near Mean Test RMSE: {np.mean(near_test_rmse_list_cpu)}")

# Sample standard deviation
print(f"Near STDV Test RMSE: {np.std(near_test_rmse_list_cpu, ddof=1)}")

# FAR
# Convert RMSE tensors to CPU and then to Python numbers before calculating mean and std
far_test_rmse_list_cpu = [rmse.cpu().item() for rmse in far_test_rmse_list]
print(f"Far Mean Test RMSE: {np.mean(far_test_rmse_list_cpu)}")

# Sample standard deviation
print(f"Far STDV Test RMSE: {np.std(far_test_rmse_list_cpu, ddof=1)}")

Near Mean Test RMSE: 0.01681385226547718
Near STDV Test RMSE: 0.0001144491020746903
Far Mean Test RMSE: 0.031377943139523266
Far STDV Test RMSE: 0.0001140003885566252
